# Prep 1 · NumPy → JAX

**Time:** about an hour. **Needs:** `pixi install` done, nothing else.

The whole project is written in [JAX](https://docs.jax.dev/). If you can write NumPy you already know
90 % of it. This notebook covers the other 10 %: five habits that are different, and one bridge into the
repo — your first two milestones.

Run every cell, read every comment, and do the four exercises. Each exercise has a **check** cell:
when it runs without an `AssertionError`, you're done.

In [ ]:
import numpy as np
import jax
import jax.numpy as jnp

print("JAX", jax.__version__, "· devices:", jax.devices())

## 1. `jnp` is `np`

Almost every NumPy function exists under `jax.numpy` with the same name and signature. Arrays convert
back and forth freely. Two differences to notice: JAX defaults to `float32` (fine for us — the images
are `float32` too), and a JAX array lives on whatever device JAX found (CPU here; the GPU at the school).

In [ ]:
x_np = np.linspace(0, 1, 5)
x = jnp.asarray(x_np)          # NumPy -> JAX
print(type(x), x.dtype)
print(jnp.sin(x) ** 2 + jnp.cos(x) ** 2)   # same maths, same names
print(np.asarray(x))           # JAX -> NumPy (for matplotlib, scikit-image, ...)

## 2. Arrays are immutable

In NumPy you write `x[0] = 1`. In JAX that raises an error: arrays are values, not buffers. Instead you
ask for a *new* array with the change applied: `x.at[0].set(1)`. This is what makes JAX able to
differentiate and compile your code — every operation is a pure function of its inputs.

In [ ]:
x = jnp.zeros(4)
try:
    x[0] = 1.0
except TypeError as e:
    print("NumPy habit ->", type(e).__name__, ":", str(e)[:60], "...")

y = x.at[0].set(1.0)           # the JAX way: a new array
print("x is unchanged:", x)
print("y:", y)
print("whole columns at once:", jnp.zeros((3, 6)).at[:, ::2].set(1.0))

### Exercise 1 — a mask with `.at`

Build an `(8, 8)` array of zeros and ones in which a *column* is either fully on or fully off:
every 4th column (columns 0 and 4) **and** the two central columns (3 and 4) are on; everything else is off.
This is exactly the shape of the k-space masks you'll build in `masks.py`.

In [ ]:
mask = jnp.zeros((8, 8))
# YOUR CODE HERE: turn on every 4th column, then columns 3 and 4, using .at[...].set(...)
...
mask

In [ ]:
# check
assert mask.shape == (8, 8)
assert float(mask.sum()) == 24.0, "3 columns x 8 rows should be on"
assert bool(jnp.all(mask == mask[0][None, :])), "every row must be identical (whole columns on/off)"
print("exercise 1 OK")

## 3. Randomness is explicit

NumPy hides a global random state; JAX makes you pass a **key**. The same key always gives the same
numbers (good: reproducible), so to get *different* numbers you `split` a key into new ones. Rule of
thumb: never reuse a key; split it and hand out the pieces.

In [ ]:
key = jax.random.PRNGKey(0)
print(jax.random.normal(key, (3,)))
print(jax.random.normal(key, (3,)), "<- same key, same numbers")

key, sub = jax.random.split(key)        # the idiom: keep `key`, spend `sub`
print(jax.random.normal(sub, (3,)), "<- a fresh subkey, fresh numbers")

### Exercise 2 — two independent draws

From one root key, produce two *different* vectors `a` and `b` of 1000 standard-normal samples.

In [ ]:
root = jax.random.PRNGKey(42)
# YOUR CODE HERE: split `root`, then draw `a` and `b` (shape (1000,)) from the two pieces
...

In [ ]:
# check
assert a.shape == b.shape == (1000,)
assert not bool(jnp.allclose(a, b)), "a and b must differ - did you reuse a key?"
assert abs(float(a.mean())) < 0.15 and abs(float(b.mean())) < 0.15
assert abs(float(jnp.corrcoef(a, b)[0, 1])) < 0.1, "a and b should be independent"
print("exercise 2 OK")

## 4. `jax.grad` — derivatives for free

Give `jax.grad` a function that returns a scalar and it returns a function that computes the gradient
with respect to the first argument. This is the engine under everything: the VAE is trained with it,
and NumPyro's MAP and NUTS use it to move through the posterior.

In [ ]:
def f(x):
    return jnp.sum(x ** 2)         # a scalar

x = jnp.array([1.0, -2.0, 3.0])
print("f(x) =", f(x))
print("grad f(x) =", jax.grad(f)(x), " (expected 2x =", 2 * x, ")")

### Exercise 3 — the gradient of a mean-squared error

Write `mse(x, y) = mean((x - y)^2)` and use `jax.grad` to get its gradient with respect to `x`.
Then verify it against the formula `2 (x - y) / n`.

In [ ]:
def mse(x, y):
    # YOUR CODE HERE
    ...

x = jnp.array([0.5, 1.5, -1.0, 2.0])
y = jnp.array([0.0, 1.0, -2.0, 2.0])
g = ...            # YOUR CODE HERE: jax.grad of mse with respect to x
g_formula = ...    # YOUR CODE HERE: 2 (x - y) / n

In [ ]:
# check
assert g is not ... and g_formula is not ..., "fill in the exercise above first"
assert g.shape == x.shape
assert bool(jnp.allclose(g, g_formula, atol=1e-6)), "gradient does not match 2(x - y)/n"
print("exercise 3 OK ·", g)

## 5. `jit` and `vmap`

`jax.jit` compiles a function the first time you call it (slow once, fast forever after). `jax.vmap`
turns a function written for *one* example into one that works on a *batch*, without you writing a loop.
The repo uses both constantly: `vmap` to score a batch of images, `jit` around every training step.

In [ ]:
import time

def slow_norm(x):
    return jnp.sqrt(jnp.sum(x ** 2))

fast_norm = jax.jit(slow_norm)
big = jnp.ones((2000, 2000))
t = time.perf_counter(); fast_norm(big).block_until_ready(); print(f"first call (compiles): {time.perf_counter()-t:.3f}s")
t = time.perf_counter(); fast_norm(big).block_until_ready(); print(f"second call:           {time.perf_counter()-t:.4f}s")

per_image_mean = lambda img: img.mean()             # written for ONE image ...
stack = jnp.arange(5 * 4 * 4, dtype=jnp.float32).reshape(5, 4, 4)
print("vmap over a stack of 5 images:", jax.vmap(per_image_mean)(stack))   # ... applied to five

### Exercise 4 — per-image error with `vmap`

`imgs` and `refs` are stacks of five `16 × 16` images. Using `mse` from exercise 3 and `jax.vmap`,
compute the five per-image errors *without a Python loop*. (Hint: `jax.vmap(mse)(imgs, refs)` — `vmap`
maps over the leading axis of every argument.)

In [ ]:
key = jax.random.PRNGKey(1)
k1, k2 = jax.random.split(key)
refs = jax.random.uniform(k1, (5, 16, 16))
imgs = refs + 0.1 * jax.random.normal(k2, (5, 16, 16))

errors = ...     # YOUR CODE HERE
errors

In [ ]:
# check
assert errors is not ..., "fill in the exercise above first"
loop_version = jnp.array([mse(imgs[i], refs[i]) for i in range(5)])
assert errors.shape == (5,)
assert bool(jnp.allclose(errors, loop_version, atol=1e-6))
print("exercise 4 OK")

## 6. Bridge to the repo — your first two milestones

Open `src/mrigen/metrics.py`. Two functions are `TODO`s:

- `psnr(gt, pred, data_range)` = `10 · log10(data_range² / MSE)` — peak signal-to-noise ratio in dB, higher is better;
- `nmse(gt, pred)` = `‖pred − gt‖² / ‖gt‖²` — normalised MSE, lower is better.

They are plain NumPy (the metrics run on CPU after reconstruction), each is two lines, and each has a
test. Implement them, then run the cell below (it reloads the module so you don't need to restart), and
run `pixi run test` in a terminal — two of the skipped tests should now pass. `pixi run milestones`
will say `2/8`.

In [ ]:
import importlib
import mrigen.metrics
importlib.reload(mrigen.metrics)
from mrigen.metrics import nmse, psnr

gt = np.random.default_rng(0).random((32, 32))
noisy = gt + 0.05 * np.random.default_rng(1).standard_normal((32, 32))
try:
    print(f"PSNR(gt, gt)    = {psnr(gt, gt, data_range=1.0):.1f} dB   (should be huge / inf)")
    print(f"PSNR(gt, noisy) = {psnr(gt, noisy, data_range=1.0):.1f} dB   (about 26 dB)")
    print(f"NMSE(gt, noisy) = {nmse(gt, noisy):.4f}      (about 0.007)")
    assert psnr(gt, gt, data_range=1.0) > 80 and nmse(gt, gt) < 1e-12
    assert 24 < psnr(gt, noisy, data_range=1.0) < 28
    print("milestones 1 and 2 OK - now run `pixi run test`")
except NotImplementedError as e:
    print("Not yet:", e)
    print("-> implement psnr and nmse in src/mrigen/metrics.py, then re-run this cell")

## Done when

- all four checks print OK;
- `pixi run milestones` reports `2/8`;
- you can say in one sentence each what `.at[].set()`, `jax.random.split`, `jax.grad` and `jax.vmap` do.

Next: **Prep 2 · Fourier transforms and k-space.**